In [1]:
import json

with open("./datasets/mocap/dataset_mocap_cycles_split.json", "r") as file:
    mocap_cycles = json.load(file)

with open("./datasets/yolo/dataset_yolo_triang_cycles_split.json", "r") as file:
    yolo_cycles = json.load(file)

with open("./datasets/yolo/dataset_yolo_triang_best_cameras_cycles_split.json", "r") as file:
    yolo_best_cams_cycles = json.load(file)

with open("./datasets/yolo/dataset_yolo_triang_cycles_split_smoothed.json", "r") as file:
    yolo_cycles_smoothed = json.load(file)

with open("./datasets/yolo/dataset_yolo_triang_best_cameras_cycles_split_smoothed.json", "r") as file:
    yolo_best_cams_cycles_smoothed = json.load(file)

In [2]:
import re

text = "p32s1c3"
pattern = r"p(\d{1,2})s(\d{1,2})c(\d{1,2})"

match = re.search(pattern, text)
if match:
    p_val, s_val, c_val = match.groups()
    print(f"P: {p_val}, S: {s_val}, C: {c_val}")


P: 32, S: 1, C: 3


In [3]:
from utils.gait_parameters_extractor_raw import GaitParametersExtractorRaw
from utils.gait_parameters_extractor import CoordinatesIdx
import re

pattern = r"p(\d{1,2})s(\d{1,2})c(\d{1,2})"

combined_participants = []
combined_sequences_parameters = []

for sequence_key, sequence_joint_positions in mocap_cycles.items():
    print(sequence_key, end=' | ')
    gpe_raw = GaitParametersExtractorRaw(sequence_joint_positions, coordinates_idx=CoordinatesIdx(2, 0, 1))
    sequence_parameters = gpe_raw.get_gait_parameters()
    match = re.search(pattern, sequence_key)
    participant, _, _ = match.groups()

    combined_participants.append(int(participant))
    combined_sequences_parameters.append(sequence_parameters)


p1s1c0 | p1s1c1 | p1s1c2 | p1s2c0 | p1s2c1 | p1s3c0 | p1s3c1 | p1s3c2 | p1s4c0 | p1s4c1 | p2s1c0 | p2s1c1 | p2s1c2 | p2s2c0 | p2s2c1 | p2s2c2 | p2s3c0 | p2s3c1 | p2s3c2 | p2s4c0 | p2s4c1 | p2s4c2 | p3s1c0 | p3s1c1 | p3s1c2 | p3s2c0 | p3s2c1 | p3s2c2 | p3s3c0 | p3s3c1 | p3s3c2 | p3s4c0 | p3s4c1 | p3s4c2 | p4s1c0 | p4s1c1 | p4s2c0 | p4s2c1 | p4s3c0 | p4s3c1 | p4s4c0 | p4s4c1 | p5s1c0 | p5s1c1 | p5s1c2 | p5s2c0 | p5s2c1 | p5s2c2 | p5s3c0 | p5s3c1 | p5s3c2 | p5s4c0 | p5s4c1 | p5s4c2 | p6s1c0 | p6s1c1 | p6s2c0 | p6s2c1 | p6s2c2 | p6s3c0 | p6s3c1 | p6s3c2 | p6s4c0 | p6s4c1 | p6s4c2 | p7s1c0 | p7s1c1 | p7s1c2 | p7s2c0 | p7s2c1 | p7s2c2 | p7s3c0 | p7s3c1 | p7s4c0 | p7s4c1 | p7s4c2 | p8s1c0 | p8s1c1 | p8s1c2 | p8s2c0 | p8s2c1 | p8s3c0 | p8s3c1 | p8s4c0 | p8s4c1 | p9s1c0 | p9s1c1 | p9s1c2 | p9s2c0 | p9s2c1 | p9s3c0 | p9s3c1 | p9s4c0 | p9s4c1 | p9s4c2 | p10s1c0 | p10s1c1 | p10s1c2 | p10s2c0 | p10s2c1 | p10s2c2 | p10s3c0 | p10s3c1 | p10s3c2 | p10s4c0 | p10s4c1 | p10s4c2 | p11s1c0 | p11s1c1 | p11s1

In [4]:
len(combined_participants)

402

In [5]:
len(combined_sequences_parameters)

402

In [6]:
def count_participants_samples(participants):
    for i in range(1, 33):
        print(f"{i} -> {participants.count(i)} samples")

count_participants_samples(combined_participants)

1 -> 10 samples
2 -> 12 samples
3 -> 12 samples
4 -> 8 samples
5 -> 12 samples
6 -> 11 samples
7 -> 11 samples
8 -> 9 samples
9 -> 10 samples
10 -> 12 samples
11 -> 10 samples
12 -> 11 samples
13 -> 9 samples
14 -> 12 samples
15 -> 8 samples
16 -> 11 samples
17 -> 11 samples
18 -> 14 samples
19 -> 10 samples
20 -> 11 samples
21 -> 11 samples
22 -> 12 samples
23 -> 8 samples
24 -> 10 samples
25 -> 7 samples
26 -> 16 samples
27 -> 19 samples
28 -> 26 samples
29 -> 24 samples
30 -> 19 samples
31 -> 23 samples
32 -> 13 samples


In [7]:
from utils.torch_siamese_raw import SiameseGaitDatasetRaw, SiameseNetworkLSTM, SiameseNetworkConv1D, ContrastiveLoss, compute_similarity
import torch
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
)



learning_rate = 1e-4
batch_size = 32
n_epochs = 10

train_dataset = SiameseGaitDatasetRaw(
    selected_participants = list(range(9, 33)),
    all_participants=combined_participants,
    features=combined_sequences_parameters
)

test_dataset = SiameseGaitDatasetRaw(
    selected_participants = list(range(9)),
    all_participants=combined_participants,
    features=combined_sequences_parameters
)

print("Train dataset size: ", len(train_dataset))
print("Test dataset size: ", len(test_dataset))

model = SiameseNetworkLSTM()
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(n_epochs):
    train_dataset.regenerate_pairs()
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True
    )

    model.train()
    train_loss = 0
    for x1, x2, label in train_loader:
        out1, out2 = model(x1, x2)
        loss = criterion(out1, out2, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    print(
        f"Epoch {epoch + 1}, Train Loss: {train_loss / len(train_loader)}" 
    )

model.eval()

Train dataset size:  4502
Test dataset size:  834
Epoch 1, Train Loss: 0.23162002481044608
Epoch 2, Train Loss: 0.07862926514964577
Epoch 3, Train Loss: 0.045930708751927875
Epoch 4, Train Loss: 0.03270718732069359
Epoch 5, Train Loss: 0.02940433935744437
Epoch 6, Train Loss: 0.01975246637247186
Epoch 7, Train Loss: 0.019514533752205628
Epoch 8, Train Loss: 0.01803660035093731
Epoch 9, Train Loss: 0.01734014748965542
Epoch 10, Train Loss: 0.012620753734141376


SiameseNetworkLSTM(
  (lstm): LSTM(16, 64, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=64, out_features=10, bias=True)
)

In [8]:
y_true = []
y_pred = []
threshold = 0.06

for ptcpt_1, ptcpt_2, label in test_dataset.data:
    y_true.append(label.item() == 0)
    y_pred.append(
        compute_similarity(ptcpt_1, ptcpt_2, model).item() < threshold
    )

print("Motion capture")

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Recall: ", recall)

cm = confusion_matrix(y_true, y_pred, labels=[True, False])
class_names = ["True", "False"]
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print("Confusion matrix: \n", cm_df)

Motion capture
Accuracy:  0.9412470023980816
Precision:  0.9401913875598086
Recall:  0.9424460431654677
Confusion matrix: 
        True  False
True    393     24
False    25    392


In [9]:
from utils.gait_parameters_extractor_raw import GaitParametersExtractorRaw
from utils.gait_parameters_extractor import CoordinatesIdx
import re

pattern = r"p(\d{1,2})s(\d{1,2})c(\d{1,2})"

combined_participants = []
combined_sequences_parameters = []

for sequence_key, sequence_joint_positions in yolo_cycles.items():
    print(sequence_key, end=' | ')
    gpe_raw = GaitParametersExtractorRaw(sequence_joint_positions, coordinates_idx=CoordinatesIdx(0, 1, 2))
    sequence_parameters = gpe_raw.get_gait_parameters()
    match = re.search(pattern, sequence_key)
    participant, _, _ = match.groups()

    combined_participants.append(int(participant))
    combined_sequences_parameters.append(sequence_parameters)


p1s1c0 | p1s1c1 | p1s2c0 | p1s2c1 | p1s3c0 | p1s3c1 | p1s4c0 | p1s4c1 | p2s1c0 | p2s1c1 | p2s1c2 | p2s2c0 | p2s2c1 | p2s2c2 | p2s3c0 | p2s3c1 | p2s3c2 | p2s4c0 | p2s4c1 | p2s4c2 | p3s1c0 | p3s1c1 | p3s1c2 | p3s2c0 | p3s2c1 | p3s3c0 | p3s3c1 | p3s3c2 | p3s4c0 | p3s4c1 | p4s1c0 | p4s1c1 | p4s2c0 | p4s2c1 | p4s3c0 | p4s3c1 | p4s4c0 | p4s4c1 | p5s1c0 | p5s1c1 | p5s1c2 | p5s2c0 | p5s2c1 | p5s2c2 | p5s2c3 | p5s3c0 | p5s3c1 | p5s3c2 | p5s4c0 | p5s4c1 | p5s4c2 | p6s1c0 | p6s1c1 | p6s2c0 | p6s2c1 | p6s2c2 | p6s3c0 | p6s3c1 | p6s3c2 | p6s4c0 | p6s4c1 | p7s1c0 | p7s1c1 | p7s1c2 | p7s2c0 | p7s2c1 | p7s2c2 | p7s3c0 | p7s3c1 | p7s4c0 | p7s4c1 | p8s1c0 | p8s1c1 | p8s1c2 | p8s2c0 | p8s2c1 | p8s3c0 | p8s3c1 | p8s3c2 | p8s4c0 | p9s1c0 | p9s1c1 | p9s2c0 | p9s2c1 | p9s2c2 | p9s3c0 | p9s3c1 | p9s4c0 | p9s4c1 | p10s1c0 | p10s1c1 | p10s1c2 | p10s2c0 | p10s2c1 | p10s3c0 | p10s3c1 | p10s4c0 | p10s4c1 | p11s1c0 | p11s1c1 | p11s1c2 | p11s2c0 | p11s2c1 | p11s3c0 | p11s3c1 | p11s3c2 | p11s4c0 | p12s1c0 | p12s1c1 |

In [10]:
count_participants_samples(combined_participants)

1 -> 8 samples
2 -> 12 samples
3 -> 10 samples
4 -> 8 samples
5 -> 13 samples
6 -> 10 samples
7 -> 10 samples
8 -> 9 samples
9 -> 9 samples
10 -> 9 samples
11 -> 9 samples
12 -> 10 samples
13 -> 10 samples
14 -> 8 samples
15 -> 8 samples
16 -> 8 samples
17 -> 10 samples
18 -> 12 samples
19 -> 6 samples
20 -> 12 samples
21 -> 11 samples
22 -> 11 samples
23 -> 6 samples
24 -> 7 samples
25 -> 5 samples
26 -> 17 samples
27 -> 19 samples
28 -> 26 samples
29 -> 22 samples
30 -> 18 samples
31 -> 18 samples
32 -> 11 samples


In [11]:
learning_rate = 1e-4
batch_size = 32
n_epochs = 10

train_dataset = SiameseGaitDatasetRaw(
    selected_participants = list(range(9, 33)),
    all_participants=combined_participants,
    features=combined_sequences_parameters
)

test_dataset = SiameseGaitDatasetRaw(
    selected_participants = list(range(9)),
    all_participants=combined_participants,
    features=combined_sequences_parameters
)

print("Train dataset size: ", len(train_dataset))
print("Test dataset size: ", len(test_dataset))

model = SiameseNetworkLSTM()
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(n_epochs):
    train_dataset.regenerate_pairs()
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True
    )

    model.train()
    train_loss = 0
    for x1, x2, label in train_loader:
        out1, out2 = model(x1, x2)
        loss = criterion(out1, out2, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    print(
        f"Epoch {epoch + 1}, Train Loss: {train_loss / len(train_loader)}" 
    )

model.eval()

Train dataset size:  3708
Test dataset size:  742
Epoch 1, Train Loss: 0.2778644358803486
Epoch 2, Train Loss: 0.2011781462693009
Epoch 3, Train Loss: 0.15997343805843386
Epoch 4, Train Loss: 0.12687078111901365
Epoch 5, Train Loss: 0.10700041675490551
Epoch 6, Train Loss: 0.08788986648593483
Epoch 7, Train Loss: 0.07135009823431229
Epoch 8, Train Loss: 0.06342979054898024
Epoch 9, Train Loss: 0.057180006924117434
Epoch 10, Train Loss: 0.04986242535684643


SiameseNetworkLSTM(
  (lstm): LSTM(16, 64, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=64, out_features=10, bias=True)
)

In [12]:
y_true = []
y_pred = []
threshold = 0.06

for ptcpt_1, ptcpt_2, label in test_dataset.data:
    y_true.append(label.item() == 0)
    y_pred.append(
        compute_similarity(ptcpt_1, ptcpt_2, model).item() < threshold
    )


print("Triangulation all cameras")

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Recall: ", recall)

cm = confusion_matrix(y_true, y_pred, labels=[True, False])
class_names = ["True", "False"]
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print("Confusion matrix: \n", cm_df)

Triangulation all cameras
Accuracy:  0.5943396226415094
Precision:  0.972972972972973
Recall:  0.1940700808625337
Confusion matrix: 
        True  False
True     72    299
False     2    369


In [13]:
from utils.gait_parameters_extractor_raw import GaitParametersExtractorRaw
from utils.gait_parameters_extractor import CoordinatesIdx
import re

pattern = r"p(\d{1,2})s(\d{1,2})c(\d{1,2})"

combined_participants = []
combined_sequences_parameters = []

for sequence_key, sequence_joint_positions in yolo_best_cams_cycles.items():
    print(sequence_key, end=' | ')
    gpe_raw = GaitParametersExtractorRaw(sequence_joint_positions, coordinates_idx=CoordinatesIdx(0, 1, 2))
    sequence_parameters = gpe_raw.get_gait_parameters()
    match = re.search(pattern, sequence_key)
    participant, _, _ = match.groups()

    combined_participants.append(int(participant))
    combined_sequences_parameters.append(sequence_parameters)


p1s1c0 | p1s1c1 | p1s2c0 | p1s2c1 | p1s3c0 | p1s3c1 | p1s3c2 | p1s4c0 | p1s4c1 | p2s2c0 | p2s2c1 | p2s2c2 | p2s3c0 | p2s3c1 | p2s3c2 | p2s4c0 | p2s4c1 | p2s4c2 | p3s1c0 | p3s1c1 | p3s1c2 | p3s2c0 | p3s2c1 | p3s3c0 | p3s3c1 | p3s3c2 | p3s4c0 | p3s4c1 | p4s1c0 | p4s1c1 | p4s2c0 | p4s2c1 | p4s3c0 | p4s3c1 | p4s4c0 | p4s4c1 | p5s1c0 | p5s1c1 | p5s1c2 | p5s2c0 | p5s2c1 | p5s2c2 | p5s2c3 | p5s3c0 | p5s3c1 | p5s3c2 | p5s4c0 | p5s4c1 | p5s4c2 | p6s1c0 | p6s1c1 | p6s2c0 | p6s2c1 | p6s2c2 | p6s3c0 | p6s3c1 | p6s3c2 | p6s4c0 | p7s1c0 | p7s1c1 | p7s1c2 | p7s2c0 | p7s2c1 | p7s2c2 | p7s3c0 | p7s3c1 | p8s1c0 | p8s1c1 | p8s1c2 | p8s2c0 | p8s2c1 | p8s2c2 | p8s3c0 | p8s3c1 | p8s3c2 | p8s4c0 | p8s4c1 | p9s1c0 | p9s1c1 | p9s2c0 | p9s2c1 | p9s2c2 | p9s3c0 | p9s3c1 | p9s4c0 | p9s4c1 | p10s1c0 | p10s1c1 | p10s2c0 | p10s2c1 | p10s3c0 | p10s3c1 | p10s4c0 | p10s4c1 | p11s1c0 | p11s1c1 | p11s1c2 | p11s2c0 | p11s2c1 | p11s3c0 | p11s3c1 | p11s4c0 | p12s1c0 | p12s1c1 | p12s1c2 | p12s2c0 | p12s2c1 | p12s3c0 | p12s3c

In [14]:
learning_rate = 1e-4
batch_size = 32
n_epochs = 10

train_dataset = SiameseGaitDatasetRaw(
    selected_participants = list(range(9, 33)),
    all_participants=combined_participants,
    features=combined_sequences_parameters
)

test_dataset = SiameseGaitDatasetRaw(
    selected_participants = list(range(9)),
    all_participants=combined_participants,
    features=combined_sequences_parameters
)

print("Train dataset size: ", len(train_dataset))
print("Test dataset size: ", len(test_dataset))

model = SiameseNetworkLSTM()
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(n_epochs):
    train_dataset.regenerate_pairs()
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True
    )

    model.train()
    train_loss = 0
    for x1, x2, label in train_loader:
        out1, out2 = model(x1, x2)
        loss = criterion(out1, out2, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    print(
        f"Epoch {epoch + 1}, Train Loss: {train_loss / len(train_loader)}" 
    )

model.eval()

Train dataset size:  3598
Test dataset size:  684
Epoch 1, Train Loss: 0.30472082585360094
Epoch 2, Train Loss: 0.18557483458940963
Epoch 3, Train Loss: 0.10693277369162678
Epoch 4, Train Loss: 0.0700027346446187
Epoch 5, Train Loss: 0.05168788669119894
Epoch 6, Train Loss: 0.040533626171867405
Epoch 7, Train Loss: 0.035406307276107565
Epoch 8, Train Loss: 0.026796464717625517
Epoch 9, Train Loss: 0.02391210475738729
Epoch 10, Train Loss: 0.020815177045894407


SiameseNetworkLSTM(
  (lstm): LSTM(16, 64, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=64, out_features=10, bias=True)
)

In [15]:
y_true = []
y_pred = []
threshold = 0.06

for ptcpt_1, ptcpt_2, label in test_dataset.data:
    y_true.append(label.item() == 0)
    y_pred.append(
        compute_similarity(ptcpt_1, ptcpt_2, model).item() < threshold
    )

print("Triangulation best cameras")

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Recall: ", recall)

cm = confusion_matrix(y_true, y_pred, labels=[True, False])
class_names = ["True", "False"]
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print("Confusion matrix: \n", cm_df)

Triangulation best cameras
Accuracy:  0.8742690058479532
Precision:  0.9637681159420289
Recall:  0.7777777777777778
Confusion matrix: 
        True  False
True    266     76
False    10    332


In [16]:
from utils.torch_siamese_raw import SiameseGaitDatasetRaw, SiameseNetworkLSTM, SiameseNetworkConv1D, ContrastiveLoss, compute_similarity


learning_rate = 1e-4
batch_size = 32
n_epochs = 10

train_dataset = SiameseGaitDatasetRaw(
    selected_participants = list(range(9, 33)),
    all_participants=combined_participants,
    features=combined_sequences_parameters
)

test_dataset = SiameseGaitDatasetRaw(
    selected_participants = list(range(9)),
    all_participants=combined_participants,
    features=combined_sequences_parameters
)

print("Train dataset size: ", len(train_dataset))
print("Test dataset size: ", len(test_dataset))

model = SiameseNetworkConv1D()
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(n_epochs):
    train_dataset.regenerate_pairs()
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True
    )

    model.train()
    train_loss = 0
    for x1, x2, label in train_loader:
        out1, out2 = model(x1, x2)
        loss = criterion(out1, out2, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    print(
        f"Epoch {epoch + 1}, Train Loss: {train_loss / len(train_loader)}" 
    )

model.eval()

Train dataset size:  3598
Test dataset size:  684
Epoch 1, Train Loss: 3.0008475248067255
Epoch 2, Train Loss: 0.06472713483777721
Epoch 3, Train Loss: 0.04487797001426199
Epoch 4, Train Loss: 0.03408202485272051
Epoch 5, Train Loss: 0.030326115907029768
Epoch 6, Train Loss: 0.022428001977701104
Epoch 7, Train Loss: 0.021893689800383507
Epoch 8, Train Loss: 0.022029038496002292
Epoch 9, Train Loss: 0.02000679947699593
Epoch 10, Train Loss: 0.025013849555424093


SiameseNetworkConv1D(
  (encoder): Sequential(
    (0): Conv1d(16, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (3): ReLU()
    (4): Conv1d(128, 256, kernel_size=(3,), stride=(1,), padding=(1,))
    (5): ReLU()
    (6): AdaptiveAvgPool1d(output_size=1)
  )
  (fc): Linear(in_features=256, out_features=10, bias=True)
)

In [17]:
y_true = []
y_pred = []
threshold = 0.3

for ptcpt_1, ptcpt_2, label in test_dataset.data:
    y_true.append(label.item() == 0)
    y_pred.append(
        compute_similarity(ptcpt_1, ptcpt_2, model).item() < threshold
    )

print("Triangulation best cameras")

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Recall: ", recall)

cm = confusion_matrix(y_true, y_pred, labels=[True, False])
class_names = ["True", "False"]
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print("Confusion matrix: \n", cm_df)

Triangulation best cameras
Accuracy:  0.8333333333333334
Precision:  0.9014084507042254
Recall:  0.7485380116959064
Confusion matrix: 
        True  False
True    256     86
False    28    314
